# MSI5001 SER

Compare seven untuned models on AESDD: DummyClassifier, Logistic Regression, KNN, SVM,
Random Forest, MLP and a small CNN. The six scikit-learn classifiers retain their
default constructor settings (including MLP's 200-iteration limit).
CNN uses two convolution layers, Adam defaults, batch size 32 and 10 epochs.

Run the eight sections in order. Classical models use 40 MFCC mean/std features;
CNN uses 64×128 log-mel images. Evaluation reports accuracy and macro F1 for one
approximately 80/20 stratified group random split and LOSO by speaker.
Alternate takes stay together in the random split. LOSO scores pool the predictions
from all held-out speakers. Every model uses the same test recordings.

## 1. Install dependency & import library

Installation is optional when the selected kernel already has the dependencies.
Run installation before imports and restart the kernel after upgrades if required.
For local NVIDIA GPU support, install a CUDA-enabled PyTorch build using the
[official selector](https://pytorch.org/get-started/locally/) in your environment;
skip the installation cell once that environment is ready.

In [ ]:
%pip install --upgrade numpy pandas scikit-learn librosa soundfile gdown torch

In [ ]:
from pathlib import Path
import re, zipfile, hashlib
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import gdown
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

## 2. Load dataset

Choose `local` or `drive`. Local input accepts a WAV directory, ZIP, or folder of ZIPs.
Drive input copies a public folder or dataset ZIP locally without mounting Drive.
The Drive sharing setting must be **Anyone with the link → Viewer**.
This section lists candidate AESDD files; audio validation happens in section 4.

In [ ]:
SOURCE = 'local'                         # 'local' or 'drive'
LOCAL_PATH = r'./AESDD'                   # WAV directory, ZIP, or folder of ZIPs
DRIVE_URL = 'PASTE_PUBLIC_FILE_OR_FOLDER_URL_HERE'

root = Path(LOCAL_PATH).expanduser().resolve()
if SOURCE == 'drive':
    root = Path('./downloaded_AESDD').resolve()
    root.mkdir(exist_ok=True)
    if '/folders/' in DRIVE_URL:
        downloaded = gdown.download_folder(DRIVE_URL, output=str(root), use_cookies=False)
    else:
        downloaded = gdown.download(DRIVE_URL, output=str(root/'dataset.zip'), use_cookies=False)
    if not downloaded:
        raise RuntimeError('Download failed. Use a public anyone-with-the-link Drive URL.')
elif SOURCE != 'local':
    raise ValueError("SOURCE must be 'local' or 'drive'.")
if not root.exists():
    raise FileNotFoundError(f'Edit LOCAL_PATH: {root}')

def unzip(path):
    destination = path.with_suffix('')
    if not (destination/'.extracted').exists():
        with zipfile.ZipFile(path) as archive:
            if any(not (destination/name).resolve().is_relative_to(destination.resolve())
                   for name in archive.namelist()):
                raise ValueError('Unsafe archive path')
            archive.extractall(destination)
        (destination/'.extracted').touch()
    return destination

if root.is_file():
    root = unzip(root)
for _ in range(3):                       # Also handles ZIPs inside a downloaded folder
    for archive in list(root.rglob('*.zip')):
        unzip(archive)

codes = {'a':'Anger', 'd':'Disgust', 'f':'Fear', 'h':'Happiness', 's':'Sadness'}
pattern = re.compile(r'([adfhs])\s*[_ -]?(\d{1,2})\s*[_ -]?\(\s*(\d+)\s*\)\s*[a-z]*', re.I)
records = []
for path in sorted(root.rglob('*.wav')):
    match = pattern.fullmatch(path.stem)
    if not match or '__MACOSX' in path.parts:
        continue
    emotion, utterance, speaker = match.groups()
    emotion, speaker = emotion.lower(), str(int(speaker))
    records.append({'path':path, 'emotion':codes[emotion], 'speaker':speaker,
                    'group':f'{speaker}_{emotion}_{int(utterance)}'})
if not records:
    raise ValueError('No AESDD WAV filenames found. Check the dataset path and filenames.')
recordings = pd.DataFrame(records)

## 3. Explore dataset

Inspect candidate files, emotion balance and recordings per speaker before cleaning.
These counts include any duplicate or damaged recordings removed in section 4.

In [ ]:
print('Candidate recordings:', len(recordings))
print('Speakers:', recordings.speaker.nunique())
print('Emotions:', recordings.emotion.nunique())
print('\nRecordings per emotion:')
print(recordings.emotion.value_counts().sort_index().to_string())
print('\nRecordings per speaker and emotion:')
print(pd.crosstab(recordings.speaker, recordings.emotion).to_string())
print('\nFirst five recordings:')
print(recordings.head().to_string(index=False))

## 4. Data preprocessing

Remove byte-identical duplicates and skip damaged, empty, invalid or silent WAVs.
Convert audio to mono and resample to 16 kHz. Extract 20 MFCC means and standard
deviations for classical ML, plus fixed-size log-mel images for CNN. These operations
are performed independently per recording. Learned scaling is deferred to section 6.

In [ ]:
rows, features, images, seen = [], [], [], set()
for record in recordings.to_dict('records'):
    path = record['path']
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest in seen:
        continue
    try:
        audio, sr = sf.read(path, dtype='float32', always_2d=True)
        audio = librosa.resample(audio.mean(axis=1), orig_sr=sr, target_sr=16000)
        if not len(audio) or not np.isfinite(audio).all() or np.max(np.abs(audio)) < 1e-7:
            raise ValueError('Empty, invalid or silent audio')
        mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=20)
        mel = librosa.feature.melspectrogram(y=audio, sr=16000, n_mels=64)
        mel = torch.tensor(librosa.power_to_db(mel), dtype=torch.float32)[None, None]
        image = nn.functional.interpolate(mel, size=(64,128), mode='bilinear', align_corners=False)[0].numpy()
    except (RuntimeError, ValueError, OSError) as error:
        print('Skipped:', path.name, error)
        continue
    rows.append(record)
    features.append(np.r_[mfcc.mean(axis=1), mfcc.std(axis=1)])
    images.append(image)
    seen.add(digest)
if not rows:
    raise ValueError('No valid AESDD WAVs found. Check the dataset path and filenames.')
metadata = pd.DataFrame(rows)
X, Z = np.asarray(features), np.asarray(images)
encoder = LabelEncoder()
y = encoder.fit_transform(metadata.emotion)
print('Valid recordings:', len(y), '| features:', X.shape[1], '| speakers:', metadata.speaker.nunique())

## 5. Train test split (random & LOSO)

Random: use the first fold of a shuffled five-fold stratified group split, keeping
alternate takes of the same speaker/emotion/utterance together.
LOSO: hold out one complete speaker at a time. LOSO follows speaker boundaries;
it does not stratify or move recordings between speakers.

In [ ]:
train, test = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
                   .split(X, y, groups=metadata.group))
splits = [('Random', train, test)]
splits += [('LOSO', tr, te) for tr, te in LeaveOneGroupOut().split(X, y, metadata.speaker)]
split_summary = pd.DataFrame([
    {'Split':setting, 'Train':len(train), 'Test':len(test),
     'Test speakers':','.join(metadata.speaker.iloc[test].unique())}
    for setting, train, test in splits
])
print(split_summary.to_string(index=False))

## 6. Feature scaling

Fit a fresh StandardScaler on each fold's **training MFCC features** and use it to
transform that fold's training and test features. Normalize CNN images using one
global mean and standard deviation computed from that fold's **training images**.
Test recordings never contribute to fitted scaling statistics.

In [ ]:
scaled_splits = []
for setting, train, test in splits:
    scaler = StandardScaler().fit(X[train])
    mean, std = Z[train].mean(), max(float(Z[train].std()), 1e-8)
    scaled_splits.append({
        'setting':setting, 'train':train, 'test':test,
        'X_train':scaler.transform(X[train]), 'X_test':scaler.transform(X[test]),
        'Z_train':(Z[train]-mean)/std, 'Z_test':(Z[test]-mean)/std,
        'y_train':y[train], 'y_test':y[test]
    })
print('Scaled splits:', len(scaled_splits))

## 7. Model training

Train fresh instances of all seven models for every split. Only training arrays
are passed into fitting. CNN uses CUDA automatically when available; the six
scikit-learn models use CPU. No hyperparameter tuning or early stopping is used.
Convergence warnings remain visible for models using their default iteration limit.

In [ ]:
EPOCHS, BATCH_SIZE = 10, 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CNN device:', DEVICE)
models = {'DummyClassifier':DummyClassifier(), 'Logistic Regression':LogisticRegression(),
          'KNN':KNeighborsClassifier(), 'SVM':SVC(),
          'Random Forest':RandomForestClassifier(), 'MLP':MLPClassifier()}

def train_cnn(images, labels):
    torch.manual_seed(42)
    inputs = torch.tensor(images)
    targets = torch.tensor(labels, dtype=torch.long)
    loader = DataLoader(TensorDataset(inputs, targets), batch_size=BATCH_SIZE, shuffle=True)
    model = nn.Sequential(
        nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32,len(encoder.classes_))
    ).to(DEVICE)
    optimizer, loss_fn = torch.optim.Adam(model.parameters()), nn.CrossEntropyLoss()
    for _ in range(EPOCHS):
        model.train()
        for batch, labels in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(batch.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
    return model

trained_splits = []
for fold in scaled_splits:
    print(fold['setting'], '| test speakers:',
          ','.join(metadata.speaker.iloc[fold['test']].unique()))
    fitted = {}
    for name, estimator in models.items():
        np.random.seed(42)
        fitted[name] = clone(estimator).fit(fold['X_train'], fold['y_train'])
    fitted['CNN'] = train_cnn(fold['Z_train'], fold['y_train'])
    trained_splits.append(fitted)

## 8. Model prediction and evaluation

Predict on held-out test arrays after training. Report accuracy and macro F1 for
the random split and pooled LOSO predictions, using all five emotion labels.

In [ ]:
predictions = []
for fold, fitted in zip(scaled_splits, trained_splits):
    for name, model in fitted.items():
        if name == 'CNN':
            model.eval()
            with torch.no_grad():
                predicted = np.concatenate([
                    model(torch.tensor(fold['Z_test'][i:i+64]).to(DEVICE))
                    .argmax(1).cpu().numpy() for i in range(0,len(fold['Z_test']),64)
                ])
        else:
            predicted = model.predict(fold['X_test'])
        predictions.append((fold['setting'], name, fold['y_test'], predicted))

results = []
for name in [*models, 'CNN']:
    row = {'Model':name}
    for setting in ['Random', 'LOSO']:
        selected = [p for p in predictions if p[0]==setting and p[1]==name]
        actual = np.concatenate([p[2] for p in selected])
        predicted = np.concatenate([p[3] for p in selected])
        row[f'{setting} accuracy'] = accuracy_score(actual, predicted)
        row[f'{setting} macro F1'] = f1_score(actual, predicted, average='macro',
                                            labels=np.arange(len(encoder.classes_)), zero_division=0)
    results.append(row)
summary = pd.DataFrame(results).set_index('Model')
print(summary.round(3).to_string())

Scores are fractions from 0 to 1. Package versions, initialization and device
can change results. Outputs have been cleared after reorganizing the workflow;
run all sections to generate results for your selected environment.

References: [gdown](https://github.com/wkentaro/gdown),
[scikit-learn](https://scikit-learn.org/stable/supervised_learning.html),
[PyTorch installation](https://pytorch.org/get-started/locally/).